# Gaussian Mixture Modelling - Error Deconvolution

In this notebook I shall demonstrate how to use the machinery defined in this rust crate 

In [ ]:
import essrs
import math
import numpy as np
from scipy.stats import multivariate_normal
import matplotlib.pyplot as plt
import time
from typing import List

In [ ]:
N_SAMPLES: int = 250
N_BURN_IN: int = 200

In [ ]:
N: List[int] = [2000, 2000, 2000]

# Mean and covariance
MU: List[np.ndarray] = [
    np.array([-2.3, 2.7]),
    np.array([3.0, 1.5]),
    np.array([7.0, 2.1])
]

X_SIGMA: List[float] = [2.0, 1.7, 4.2]
Y_SIGMA: List[float] = [3.5, 1.8, 0.9]
CORR: List[float] = [0.55, -0.4, 0.02]

L = []

for xsig, ysig, c in zip(X_SIGMA, Y_SIGMA, CORR):

    # Cholesky decomposition: Sigma = L @ L.T
    L.append(
        np.linalg.cholesky(
            np.array([
                [xsig**2, xsig * ysig * c],
                [xsig * ysig * c, ysig**2]
            ])
        )
    )

In [ ]:
DATA_X = np.empty(shape=(0,2))

for n, mu, lchol in zip(N, MU, L):

    # Sample N independent standard 2D Gaussians
    Z = np.random.randn(n, 2)

    # Transform to N(mu, Sigma)
    DATA_X = np.concatenate((DATA_X, mu + Z @ lchol.T), axis=0)

DATA_X = DATA_X[np.argsort(DATA_X[:,0])]


In [ ]:
plt.figure()
plt.scatter(DATA_X[:, 0], DATA_X[:, 1], s=0.5, marker=".")
plt.show()

In [ ]:
N_COMPONENTS: int = 3
N_SUB_PARAMETERS: int = 5

N_PARAMETERS: int = (N_COMPONENTS - 1) + N_SUB_PARAMETERS * N_COMPONENTS

N_WALKERS: int = 50

In [ ]:
n = len(DATA_X)
nper = math.floor(n / N_COMPONENTS)

ics = np.random.normal(
    loc=1.0/N_COMPONENTS,
    scale=0.02,
    size=(N_WALKERS, N_COMPONENTS-1)
)

for k in range(N_COMPONENTS):
    _l = k * nper
    _r = (k+1) * nper
    mu_x = np.mean(DATA_X[_l:_r, 0])
    mu_y = np.mean(DATA_X[_l:_r, 1])
    cov_mat = np.cov(DATA_X.T)

    x_std = np.sqrt(cov_mat[0, 0])
    y_std = np.sqrt(cov_mat[1, 1])

    p = cov_mat[0, 1] / (x_std * y_std)

    ics = np.column_stack((
        ics,
        np.array([mu_x, mu_y, x_std, y_std, p]) * np.random.normal(1, 0.05, size=(N_WALKERS, N_SUB_PARAMETERS))
    ))

In [ ]:
model = essrs.model.scientific.normal_2d_gmm_deconv(
    n_components=N_COMPONENTS,
    data=np.column_stack((DATA_X, np.zeros((DATA_X.shape[0], 3))))
)

burn_move = essrs.moves.hybrid_ddc_move(one_in_n=10)
sampler_move = essrs.moves.differential_move()
config = essrs.EnsembleConfig(n_samples=N_SAMPLES, n_walkers=N_WALKERS, n_parameters=N_PARAMETERS, n_burn_in=N_BURN_IN)

start = time.perf_counter()
fit, ll = essrs.run_mcmc(
    log_likelihood=model,
    config=config,
    burn_in_move=burn_move,
    sampling_move=sampler_move,
    initial_conditions=ics
)

elapsed = time.perf_counter() - start
print(f"ESSRS took {elapsed:.3f} seconds")

In [ ]:
x = np.linspace(np.amin(DATA_X[:, 0]), np.amax(DATA_X[:, 0]), 300)
y = np.linspace(np.amin(DATA_X[:, 1]), np.amax(DATA_X[:, 1]), 300)

dxdy = (x[1] - x[0]) * (y[1] - y[0])

X, Y = np.meshgrid(x, y)


plt.figure()
plt.scatter(DATA_X[:, 0], DATA_X[:, 1], s=0.1, marker=".", color="r", alpha=0.2)

for parameters in fit[-1, :, :]:
    Z = np.zeros_like(X)
    thetas = list(parameters[:N_COMPONENTS - 1])
    thetas.append(1 - sum(thetas))
    mix = np.reshape(
        parameters[N_COMPONENTS - 1:],
        shape=(-1, 5)
    )
    for theta, phi in zip(thetas, mix):
        Zi = theta * multivariate_normal.pdf(
            np.stack([X, Y], axis=-1),
            mean=[phi[0], phi[1]],
            cov=[
                [phi[2]**2, phi[2]*phi[3]*phi[3]],
                [phi[2]*phi[3]*phi[4], phi[3]**2]
            ]
        )
        plt.contour(X, Y, Zi, alpha=0.05, cmap="Greens")

        Z += Zi


    Z /= np.sum(Z) * dxdy

    plt.contour(X, Y, Z, alpha=0.5, cmap="Blues")


plt.xlim(-7, 10)
plt.ylim(-9,11)
plt.show()